# 03 - Queries Verification

## Team Challenge SQL - E-commerce tecnológico





1. Importamos las dependencias

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from google.cloud import bigquery
from google.oauth2 import service_account

print("Imports OK")

Imports OK


In [ ]:
PROJECT_ROOT = Path(
    r"C:\Users\casco\Desktop\bootcamp\Entregas\SQLMurderMystery"
)

ENV_PATH = PROJECT_ROOT / ".env"

CREDENTIALS_PATH = (
    PROJECT_ROOT
    / "credentials"
    / "service-account.json"
)

load_dotenv(ENV_PATH)

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")

credentials = (
    service_account
    .Credentials
    .from_service_account_file(
        str(CREDENTIALS_PATH)
    )
)

client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)

print("Conexión correcta")
print("Proyecto:", PROJECT_ID)
print("Dataset:", DATASET_ID)

✅ Conexión correcta
Proyecto: 1057049326827
Dataset: amisbeauty


2. Consulta el catálogo de BigQuery para listar las tablas que existen en el dataset

In [ ]:
query = f"""
SELECT
    table_name
FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name
"""

df_tables = client.query(query).to_dataframe()

print("Tablas encontradas:")
print()

for table in df_tables["table_name"]:
    print(f" {table}")

print()
print(f"Total de tablas: {len(df_tables)}")

c:\Users\casco\Desktop\bootcamp\Entregas\SQLMurderMystery\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Tablas encontradas:

✅ categories
✅ customers
✅ order_items
✅ orders
✅ payments
✅ products
✅ reviews

Total de tablas: 7


3. Cuenta los registros que tiene cada tabla para comprobar el tamaño de cada tabla una vez cargados los datos

In [4]:
tables = [
    "customers",
    "categories",
    "products",
    "orders",
    "order_items",
    "payments",
    "reviews"
]

for table_name in tables:
    query = f"""
    SELECT COUNT(*) AS total
    FROM `{PROJECT_ID}.{DATASET_ID}.{table_name}`
    """

    result = list(client.query(query).result())
    total = result[0].total

    print(f"{table_name:15} → {total:,} registros")

customers       → 500 registros
categories      → 10 registros
products        → 70 registros
orders          → 2,000 registros
order_items     → 4,500 registros
payments        → 2,000 registros
reviews         → 317 registros


4. Obtenemos las ventas por categoría de producto

In [5]:
query = f"""
SELECT
    c.name AS category,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    SUM(oi.quantity) AS units_sold,
    ROUND(
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount)),
        2
    ) AS total_sales
FROM `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
JOIN `{PROJECT_ID}.{DATASET_ID}.products` AS p
    ON oi.product_id = p.product_id
JOIN `{PROJECT_ID}.{DATASET_ID}.categories` AS c
    ON p.category_id = c.category_id
JOIN `{PROJECT_ID}.{DATASET_ID}.orders` AS o
    ON oi.order_id = o.order_id
WHERE o.order_status != 'Cancelled'
GROUP BY c.name
ORDER BY total_sales DESC
"""

df_sales_category = client.query(query).to_dataframe()

df_sales_category

c:\Users\casco\Desktop\bootcamp\Entregas\SQLMurderMystery\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,category,total_orders,units_sold,total_sales
0,Headphones,316,688,623458.570000000
1,Storage,320,730,579430.650000000
2,Wearables,333,750,511494.950000000
3,Monitors,309,661,505247.530000000
4,Smartphones,317,704,492554.690000000
5,Cameras,296,643,465537.090000000
6,Tablets,345,763,422532.250000000
7,Mice,329,730,403281.740000000
8,Laptops,318,721,343220.280000000
9,Keyboards,307,684,319181.320000000


5. Top 10 de productos más vendidos

In [6]:
query = f"""
SELECT
    p.name AS product,
    c.name AS category,
    SUM(oi.quantity) AS units_sold,
    ROUND(
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount)),
        2
    ) AS total_sales
FROM `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
JOIN `{PROJECT_ID}.{DATASET_ID}.products` AS p
    ON oi.product_id = p.product_id
JOIN `{PROJECT_ID}.{DATASET_ID}.categories` AS c
    ON p.category_id = c.category_id
JOIN `{PROJECT_ID}.{DATASET_ID}.orders` AS o
    ON oi.order_id = o.order_id
WHERE o.order_status != 'Cancelled'
GROUP BY
    p.name,
    c.name
ORDER BY units_sold DESC
LIMIT 10
"""

df_top_products = client.query(query).to_dataframe()

df_top_products

c:\Users\casco\Desktop\bootcamp\Entregas\SQLMurderMystery\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,product,category,units_sold,total_sales
0,Samsung Galaxy Tab A9,Tablets,145,123060.270000000
1,iPhone 15 Pro,Smartphones,137,150809.920000000
2,ASUS ZenBook 14,Laptops,131,113625.290000000
3,Samsung Galaxy Watch 7,Wearables,130,88338.060000000
4,HP Pavilion 15,Laptops,128,25229.990000000
5,SanDisk Ultra 256GB,Storage,127,127742.280000000
6,Apple Watch Series 10,Wearables,125,72987.910000000
7,Samsung Galaxy Buds3,Headphones,124,94028.860000000
8,Logitech MX Master 3S,Mice,124,167271.950000000
9,AOC Gaming 24,Monitors,123,129167.620000000


6. Top 10 clientes que más dinero han gastado

In [7]:
query = f"""
SELECT
    c.customer_id,
    CONCAT(c.first_name, ' ', c.last_name) AS customer,
    c.country,
    COUNT(o.order_id) AS total_orders,
    ROUND(
        SUM(
            (
                SELECT SUM(
                    oi.quantity
                    * oi.unit_price
                    * (1 - oi.discount)
                )
                FROM `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
                WHERE oi.order_id = o.order_id
            )
        ),
        2
    ) AS total_spent
FROM `{PROJECT_ID}.{DATASET_ID}.customers` AS c
JOIN `{PROJECT_ID}.{DATASET_ID}.orders` AS o
    ON c.customer_id = o.customer_id
WHERE o.order_status != 'Cancelled'
GROUP BY
    c.customer_id,
    c.first_name,
    c.last_name,
    c.country
ORDER BY total_spent DESC
LIMIT 10
"""

df_top_customers = client.query(query).to_dataframe()

df_top_customers

c:\Users\casco\Desktop\bootcamp\Entregas\SQLMurderMystery\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,customer_id,customer,country,total_orders,total_spent
0,CUST_585B2A6A,Zaira Gaya,France,7,36165.820000000
1,CUST_16A4215D,Sol Benavides,France,10,33060.820000000
2,CUST_1918D95F,Manuelita Porras,Italy,6,30932.530000000
3,CUST_A0D6E7A4,Gisela Jaén,Germany,8,30361.560000000
4,CUST_A45BD855,Natanael Borrell,France,8,29494.990000000
5,CUST_A0207408,Concha Gálvez,Portugal,10,29178.090000000
6,CUST_9B64D180,José Antonio Torrecilla,Germany,7,28180.480000000
7,CUST_46D868B4,Silvia Ibarra,Spain,9,28039.800000000
8,CUST_F463590B,Eulalia Ferreras,United Kingdom,7,27843.530000000
9,CUST_461D06B2,Urbano Carranza,Italy,9,27837.980000000


7. Evolución mensual de las ventas: Agrupa todos los pedidos por mes y calcula cuántos pedidos hubo, cuántas unidades se vendieron y cuánto dinero se facturó cada mes

In [8]:
query = f"""
SELECT
    FORMAT_DATE('%Y-%m', o.order_date) AS month,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(oi.quantity) AS units_sold,
    ROUND(
        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount)
        ),
        2
    ) AS total_sales
FROM `{PROJECT_ID}.{DATASET_ID}.orders` AS o
JOIN `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
    ON o.order_id = oi.order_id
WHERE o.order_status != 'Cancelled'
GROUP BY month
ORDER BY month
"""

df_monthly_sales = client.query(query).to_dataframe()

df_monthly_sales

c:\Users\casco\Desktop\bootcamp\Entregas\SQLMurderMystery\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,month,total_orders,units_sold,total_sales
0,2022-06,3,22,13044.800000000
1,2022-07,2,13,6951.990000000
2,2022-08,3,13,11118.650000000
3,2022-09,4,15,11700.930000000
4,2022-10,9,37,19577.280000000
5,2022-11,2,13,9969.710000000
6,2022-12,2,7,4679.260000000
7,2023-01,2,9,7327.160000000
8,2023-02,6,32,10587.930000000
9,2023-03,6,34,21711.530000000


8. Top 10 productos más valorados

In [9]:
query = f"""
SELECT
    p.name AS product,
    c.name AS category,
    COUNT(r.review_id) AS total_reviews,
    ROUND(AVG(r.rating), 2) AS average_rating
FROM `{PROJECT_ID}.{DATASET_ID}.reviews` AS r
JOIN `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
    ON r.order_item_id = oi.order_item_id
JOIN `{PROJECT_ID}.{DATASET_ID}.products` AS p
    ON oi.product_id = p.product_id
JOIN `{PROJECT_ID}.{DATASET_ID}.categories` AS c
    ON p.category_id = c.category_id
GROUP BY
    p.name,
    c.name
HAVING total_reviews >= 3
ORDER BY average_rating DESC, total_reviews DESC
LIMIT 10
"""

df_product_ratings = client.query(query).to_dataframe()

df_product_ratings

c:\Users\casco\Desktop\bootcamp\Entregas\SQLMurderMystery\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,product,category,total_reviews,average_rating
0,Bose QuietComfort,Headphones,4,5.00
1,SanDisk Extreme SSD,Storage,3,5.00
2,Fujifilm X-S20,Cameras,3,5.00
3,MacBook Air M3,Laptops,8,4.88
4,Sony Alpha A6400,Cameras,7,4.86
5,JBL Live 770NC,Headphones,6,4.83
6,Sennheiser Momentum 4,Headphones,6,4.83
7,MacBook Pro M3,Laptops,4,4.75
8,Samsung Galaxy Watch 7,Wearables,7,4.71
9,Apple AirPods Pro 2,Headphones,6,4.67


9. Comprobación de claves foráneas 

In [ ]:
checks = {
    "products → categories": f"""
        SELECT COUNT(*) AS invalid
        FROM `{PROJECT_ID}.{DATASET_ID}.products` p
        LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.categories` c
            ON p.category_id = c.category_id
        WHERE c.category_id IS NULL
    """,

    "orders → customers": f"""
        SELECT COUNT(*) AS invalid
        FROM `{PROJECT_ID}.{DATASET_ID}.orders` o
        LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.customers` c
            ON o.customer_id = c.customer_id
        WHERE c.customer_id IS NULL
    """,

    "order_items → orders": f"""
        SELECT COUNT(*) AS invalid
        FROM `{PROJECT_ID}.{DATASET_ID}.order_items` oi
        LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.orders` o
            ON oi.order_id = o.order_id
        WHERE o.order_id IS NULL
    """,

    "order_items → products": f"""
        SELECT COUNT(*) AS invalid
        FROM `{PROJECT_ID}.{DATASET_ID}.order_items` oi
        LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.products` p
            ON oi.product_id = p.product_id
        WHERE p.product_id IS NULL
    """,

    "payments → orders": f"""
        SELECT COUNT(*) AS invalid
        FROM `{PROJECT_ID}.{DATASET_ID}.payments` p
        LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.orders` o
            ON p.order_id = o.order_id
        WHERE o.order_id IS NULL
    """,

    "reviews → order_items": f"""
        SELECT COUNT(*) AS invalid
        FROM `{PROJECT_ID}.{DATASET_ID}.reviews` r
        LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.order_items` oi
            ON r.order_item_id = oi.order_item_id
        WHERE oi.order_item_id IS NULL
    """,

    "reviews → customers": f"""
        SELECT COUNT(*) AS invalid
        FROM `{PROJECT_ID}.{DATASET_ID}.reviews` r
        LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.customers` c
            ON r.customer_id = c.customer_id
        WHERE c.customer_id IS NULL
    """
}

print("Comprobación de claves foráneas:")
print()

for relationship, query in checks.items():
    result = list(client.query(query).result())
    invalid = result[0].invalid

    status = "OK" if invalid == 0 else "ERROR"

    print(f"{status} {relationship}: {invalid}")

Comprobación de claves foráneas:

✅ OK products → categories: 0
✅ OK orders → customers: 0
✅ OK order_items → orders: 0
✅ OK order_items → products: 0
✅ OK payments → orders: 0
✅ OK reviews → order_items: 0
✅ OK reviews → customers: 0


10. Resumen de los pagos: cúantos hay, cuánto dinero suman y cuál es el importe medio de los pagos

In [11]:
query = f"""
SELECT
    payment_method,
    payment_status,
    COUNT(payment_id) AS total_payments,
    ROUND(SUM(amount), 2) AS total_amount,
    ROUND(AVG(amount), 2) AS average_payment
FROM `{PROJECT_ID}.{DATASET_ID}.payments`
GROUP BY
    payment_method,
    payment_status
ORDER BY
    payment_method,
    payment_status
"""

df_payment_summary = client.query(query).to_dataframe()

df_payment_summary

c:\Users\casco\Desktop\bootcamp\Entregas\SQLMurderMystery\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,payment_method,payment_status,total_payments,total_amount,average_payment
0,Apple Pay,Failed,48,184847.370000000,3850.990000000
1,Apple Pay,Paid,191,565900.210000000,2962.830000000
2,Apple Pay,Pending,71,168404.280000000,2371.890000000
3,Apple Pay,Refunded,71,236827.900000000,3335.600000000
4,Bank Transfer,Failed,43,127900.480000000,2974.430000000
5,Bank Transfer,Paid,179,542875.610000000,3032.820000000
6,Bank Transfer,Pending,105,328343.150000000,3127.080000000
7,Bank Transfer,Refunded,91,322249.750000000,3541.210000000
8,Credit Card,Failed,38,118583.370000000,3120.620000000
9,Credit Card,Paid,197,644776.030000000,3272.970000000


11. Analiza los tiempos de entrega: calcula cuánto tardan, de media, los pedidos en llegar desde que se realizan hasta que se entregan

In [12]:
query = f"""
SELECT
    shipping_country,
    COUNT(order_id) AS delivered_orders,
    ROUND(
        AVG(
            DATE_DIFF(
                delivered_date,
                order_date,
                DAY
            )
        ),
        2
    ) AS average_delivery_days,
    MIN(
        DATE_DIFF(
            delivered_date,
            order_date,
            DAY
        )
    ) AS min_delivery_days,
    MAX(
        DATE_DIFF(
            delivered_date,
            order_date,
            DAY
        )
    ) AS max_delivery_days
FROM `{PROJECT_ID}.{DATASET_ID}.orders`
WHERE order_status = 'Delivered'
  AND delivered_date IS NOT NULL
GROUP BY shipping_country
ORDER BY average_delivery_days
"""

df_delivery_time = client.query(query).to_dataframe()

df_delivery_time

c:\Users\casco\Desktop\bootcamp\Entregas\SQLMurderMystery\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,shipping_country,delivered_orders,average_delivery_days,min_delivery_days,max_delivery_days
0,Italy,78,6.88,2,11
1,Portugal,93,6.92,2,12
2,United Kingdom,67,6.94,2,12
3,Spain,57,7.09,2,12
4,France,46,7.13,2,12
5,Germany,74,7.19,2,12


12. Clientes que no han realizdo pedidos

In [14]:
query = f"""
SELECT
    c.customer_id,
    CONCAT(c.first_name, ' ', c.last_name) AS customer,
    c.country
FROM `{PROJECT_ID}.{DATASET_ID}.customers` AS c
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.orders` AS o
    ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
ORDER BY c.country, customer
"""

df_never_ordered = client.query(query).to_dataframe()

df_never_ordered

c:\Users\casco\Desktop\bootcamp\Entregas\SQLMurderMystery\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,customer_id,customer,country
0,CUST_C84BEF22,Adela Serrano,France
1,CUST_1D9C903E,Dalila Abella,France
2,CUST_653AADE5,Gastón Conesa,France
3,CUST_1DB2C4F1,Silvestre Cordero,France
4,CUST_7369771B,Jenny Gallart,Germany
5,CUST_3A2F3B63,Eutimio Aguilar,Italy
6,CUST_FFD78D5C,Olimpia Lerma,Italy
7,CUST_F3A2D9E2,Marcos Galvez,Portugal
8,CUST_1568BBE6,María Fernanda Uriarte,Portugal
9,CUST_12444BA1,Eli Alarcón,Spain


13. Rating medio por categoría

In [15]:
query = f"""
SELECT
    c.name AS category,
    COUNT(r.review_id) AS total_reviews,
    ROUND(AVG(r.rating), 2) AS average_rating
FROM `{PROJECT_ID}.{DATASET_ID}.reviews` AS r
JOIN `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
    ON r.order_item_id = oi.order_item_id
JOIN `{PROJECT_ID}.{DATASET_ID}.products` AS p
    ON oi.product_id = p.product_id
JOIN `{PROJECT_ID}.{DATASET_ID}.categories` AS c
    ON p.category_id = c.category_id
GROUP BY c.name
ORDER BY average_rating DESC
"""

df_category_ratings = client.query(query).to_dataframe()

df_category_ratings

c:\Users\casco\Desktop\bootcamp\Entregas\SQLMurderMystery\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,category,total_reviews,average_rating
0,Headphones,38,4.55
1,Cameras,25,4.48
2,Laptops,30,4.43
3,Wearables,27,4.37
4,Tablets,27,4.30
5,Storage,31,4.26
6,Mice,29,4.21
7,Smartphones,30,4.17
8,Monitors,40,4.10
9,Keyboards,40,4.05


14. Método de pagos utilizados

In [ ]:
query = f"""
SELECT
    payment_method,
    COUNT(payment_id) AS total_payments,
    ROUND(
        COUNT(payment_id)
        / SUM(COUNT(payment_id)) OVER () * 100,
        2
    ) AS percentage
FROM `{PROJECT_ID}.{DATASET_ID}.payments`
GROUP BY payment_method
ORDER BY total_payments DESC
"""

df_payment_methods = client.query(query).to_dataframe()

df_payment_methods